In [1]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  from google.colab import drive

  drive.mount('/content/drive') # load google drive
  os.chdir('/content/drive/My Drive/Thesis_Repository/Final__Generation') # change directory to the current working directory

In [2]:
!pip install -q aiolimiter

In [3]:
!sudo apt-get install zstd lshw

!pip install langchain_openai
!pip install langchain_ollama
!pip install langchain-google-genai

!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  pci.ids usb.ids
The following NEW packages will be installed:
  lshw pci.ids usb.ids zstd
0 upgraded, 4 newly installed, 0 to remove and 57 not upgraded.
Need to get 1,394 kB of archives.
After this operation, 4,683 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1 [322 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 1,394 kB in 0s (11.3 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based front

In [4]:
!pkill -f 'ollama serve' || true
!pkill -f 'ollama_llama_server' || true
!sleep 2

!env OLLAMA_NUM_PARALLEL=100 \
     OLLAMA_MAX_LOADED_MODELS=1 \
     OLLAMA_CONTEXT_LENGTH=4096 \
     OLLAMA_KEEP_ALIVE=5m \
     nohup ollama serve > /content/ollama_serve.log 2>&1 &

^C
^C


In [5]:
!ollama pull llama3.1:8b
!ollama pull llama3.2:3b
!ollama pull llama3.2:1b
!ollama pull mistral-nemo:12b

In [ ]:
%%writefile .env
OPENAI_API_KEY="[OPENAI API Key]"
GEMINI_API_KEY="[GEMINI API Key]"

Overwriting .env


In [3]:
import pandas as pd
import json
from pprint import pprint
import time

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
OPEN_API_KEY = os.getenv("OPEN_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") # https://ai.google.dev/gemini-api/docs/api-key?hl=ko

In [5]:
import json
import re
import random
import os
from pathlib import Path

import re
import unicodedata
# import emoji
import pandas as pd
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from typing import Any

MISSING_TEXT_VALUES = {"", "none", "nan", "null", "<na>", "n/a"}

WHITESPACE_PATTERN = re.compile(r"\s+")

def normalize_text(content) -> str:
    if isinstance(content, (list, dict, set, tuple)):
        return ""

    if content is None:
        return ""

    try:
        if pd.isna(content):
            return ""
    except (TypeError, ValueError):
        return ""

    text = str(content)

    if text.lower() in MISSING_TEXT_VALUES:
        return ""

    text = unicodedata.normalize("NFKC", text)

    # text = emoji.replace_emoji(text, replace=" ")

    text = text.lower()

    text = WHITESPACE_PATTERN.sub(" ", text).strip()

    return text

In [ ]:
## File path
FILE_PATH = 'train_test_split/train_stepverify_labeled_0.9.json' #

CONTEXT_DATA_FILE_PATH = 'data_preference_train_only/bridge_2tage__with_reference_solution_more_models/context_data__strategy_intention.json'
ALL_RESULTS_FILE_PATH = 'data_preference_train_only/bridge_2tage__with_reference_solution_more_models/preference_score__all_results.json'

# PER_CONTEXT__BRIEF_ALL_RESULTS__FILE_PATH = 'data_preference_train_train_only/bridge_2tage__with_reference_solution_more_models/preference_socre__per_context__brief.json'
# PREFERENCE_PAIRS_FILE_PATH = 'data_preference_train_train_only/bridge_2tage__with_reference_solution_more_models/preference_pairs.json'

ALL_RESULTS_DF_FILE_PATH = 'data_preference_train_only/bridge_2tage__with_reference_solution_more_models/preference_score__all_results_df.csv'
RESPONSE_PAIRS_DF_FILE_PATH = 'data_preference_train_only/bridge_2tage__with_reference_solution_more_models/response_pairs_df.csv'

## Context Data Creation
Bridging the Novice-Expert Gap via Models of Decision-Making: A Case Study on Remediating Math Mistakes ( Wang et al., 2024 )

-> z_what ( strategy ) : the definition of the pedagogy category of the first tutor turn utterance in stepverify data as it serves to directly answer the student's mistake in mathdial

-> z_why ( intention ) : the intention of the pedaogy of the first tutor turn utterance in mathdial

위 방법 안됨.
그냥 Bridging the Novice-Expert Gap via Models of Decision-Making: A Case Study on Remediating Math Mistakes ( Wang et al., 2024 ) 의 프롬프트 가져와서 strategy와 intention 따로 가져와야됨

In [7]:
def load_data(file_path=FILE_PATH):
    with open(file_path, 'r') as f:
        all_data = json.load(f)

    return all_data


def get_context_data(all_data):
    context_data = []

    for data in all_data:
        context_data.append(
            {
                'topic' : data['topic'],
                'problem' : data['problem'],
                'reference_solution': data['reference_solution'],
                'student_mistake' : data['student_incorrect_solution'],
                'error_description' : data['error_description'], # StepVerify error description -> ablation
                'error_category' : data['error_category'],
            }
        )
    return context_data

In [8]:
data = load_data(FILE_PATH)
context_data = get_context_data(data)

## 1. Srategy & Intention Selection ( Bridge )

#### Prompt

In [9]:
strategy_dict = {
    0: "explain a concept",
    1: "ask a question",
    2: "provide a hint",
    3: "provide a strategy",
    4: "provide a worked example",
    5: "provide a minor correction",
    6: "provide a similar problem",
    7: "simplify the question",
    8: "affirm the correct answer",
    9: "encourage the student",
    10: "other (specify in your reasoning)"
}

intention_dict = {
    0: "motivate the student",
    1: "get the student to elaborate their answer",
    2: "correct the student's mistake",
    3: "hint at the student's mistake",
    4: "clarify a student's misunderstanding",
    5: "help the student understand the lesson topic or solution strategy",
    6: "diagnose the student's mistake",
    7: "support the student in their thinking or problem-solving",
    8: "explain the student's mistake (e.g., what is wrong in their answer or why it is incorrect)",
    9: "signal to the student that they have solved or not solved the problem",
    10: "other (specify in your reasoning)"
}


In [ ]:
strategy_intention_system_prompt = """
you are an experienced elementary math teacher. your take is to read a student's mistake. your task is to read the student mistake. you should then determine what strategy you want to use to remediate the student's error.
"""

strategy_intention_user_prompt = """
we have a list of common strategies and intentions that teachers use, which you can pick from. we also give you the option to write in your own strategy and intention if none of the options apply.

strategies:
0. explain a concept
1. ask a question
2. provide a hint
3. provide a strategy
4. provide a worked example
5. provide a minor correction
6. provide a similar problem
7. simplify the question
8. affirm the correct answer
9. encourage the student
10. other (specify in your reasoning)

intentions:
0. motivate the student
1. get the student to elaborate their answer
2. correct the student's mistake
3. hint at the student's mistake
4. clarify a student's misunderstanding
5. help the student understand the lesson topic or solution strategy
6. diagnose the student's mistake
7. support the student in their thinking or problem-solving
8. explain the student's mistake (example: what is wrong in their answer or why is it incorrect)
9. signal to the student that they have solved or not solved the problem
10. other (specify in your reasoning)

the problem in {topic} you are student is solving is : {problem}

the reference solution is: {reference_solution}

the student made a following mistake in solving the problem: {student_mistake}

how would you remediate the student's error? pick the option number from the list of strategies and intentions.
"""

#### Generate strategy & intention ( gpt-4o )

In [26]:
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class StrategyIntention(BaseModel):
    strategy: int = Field(..., ge=0, le=10, description="Strategy option number")
    intention: int = Field(..., ge=0, le=10, description="Intention option number")


def get_strategy_intention_chain(system_prompt, user_prompt, model_name):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", user_prompt)
    ])

    model = ChatOpenAI(model=model_name, temperature=1, max_tokens=1000)

    structured_model = model.with_structured_output(StrategyIntention, method="json_schema") # PydanticOutputParser is not needed

    chain = prompt | structured_model

    return chain


async def get_context_data_with_strategy_intention_with_reference_solution(context_data, max_concurrency=50, batch_size=50, model_name="gpt-4o"):
    start_time = time.time()
    total = len(context_data)
    chain = get_strategy_intention_chain(strategy_intention_system_prompt, strategy_intention_user_prompt, model_name=model_name)

    context_data_inputs = [
        {
            "topic": data["topic"],
            "problem": data["problem"],
            "reference_solution": data["reference_solution"],
            "student_mistake": data["student_mistake"],
            "error_description": data["error_description"],
            "error_category": data["error_category"]
        }
        for data in context_data
    ]

    total_batches = (total + batch_size - 1) // batch_size

    for batch_idx, start in enumerate(range(0, total, batch_size), start=1):
        end = min(start + batch_size, total)
        context_data_batch = context_data_inputs[start:end]

        batch_start_time = time.time()
        responses = await chain.abatch(
            context_data_batch,
            config={"max_concurrency": max_concurrency})

        for i, response in enumerate(responses):
            idx = start + i
            context_data[idx]["intention"] = response.intention
            context_data[idx]["strategy"] = response.strategy

        batch_elapsed = time.time() - batch_start_time
        total_elapsed = time.time() - start_time
        print(f"[Batch {batch_idx}/{total_batches}] Completed | {end}/{total} processed | Batch: {batch_elapsed:.1f}s | Total: {total_elapsed:.1f}s", flush=True)

    return context_data

In [27]:
context_data_with_strategy_intention = None

if os.path.exists(CONTEXT_DATA_FILE_PATH):
    print(f"Loading {CONTEXT_DATA_FILE_PATH}...")
    with open(CONTEXT_DATA_FILE_PATH, 'r', encoding='utf-8') as f:
        context_data_with_strategy_intention = json.load(f)
else:
    print(f'adding strategy and intention to context data...')
    context_data_with_strategy_intention = await get_context_data_with_strategy_intention_with_reference_solution(context_data, max_concurrency=100, model_name="gpt-5.6-terra")  # gpt-5.6-terra

    os.makedirs(os.path.dirname(CONTEXT_DATA_FILE_PATH), exist_ok=True)

    with open(CONTEXT_DATA_FILE_PATH, 'w', encoding='utf-8') as f:
        print(f"Saving {CONTEXT_DATA_FILE_PATH}...")
        json.dump(context_data_with_strategy_intention, f, ensure_ascii=False, indent=2)

adding strategy and intention to context data...
[Batch 1/14] Completed | 50/693 processed | Batch: 6.6s | Total: 7.0s
[Batch 2/14] Completed | 100/693 processed | Batch: 4.5s | Total: 11.5s
[Batch 3/14] Completed | 150/693 processed | Batch: 45.4s | Total: 56.9s
[Batch 4/14] Completed | 200/693 processed | Batch: 8.3s | Total: 65.2s
[Batch 5/14] Completed | 250/693 processed | Batch: 3.7s | Total: 68.9s
[Batch 6/14] Completed | 300/693 processed | Batch: 5.6s | Total: 74.5s
[Batch 7/14] Completed | 350/693 processed | Batch: 4.6s | Total: 79.2s
[Batch 8/14] Completed | 400/693 processed | Batch: 3.9s | Total: 83.1s
[Batch 9/14] Completed | 450/693 processed | Batch: 3.4s | Total: 86.5s
[Batch 10/14] Completed | 500/693 processed | Batch: 5.4s | Total: 91.9s
[Batch 11/14] Completed | 550/693 processed | Batch: 7.7s | Total: 99.5s
[Batch 12/14] Completed | 600/693 processed | Batch: 4.7s | Total: 104.2s
[Batch 13/14] Completed | 650/693 processed | Batch: 6.0s | Total: 110.2s
[Batch 14/

## 2. Tutor Response Generation Function

Async Ainvoke  

- https://docs.ollama.com/api/openai-compatibility
- poetry add langchain langchain-openai langchain-ollama
- ollama pull llama3.1:8b   /  ollama pull llama3:8b

#### Prompt

In [28]:
generation_system_prompt = """
You are an experienced elementary mathematics tutor.

Your role is not merely to correct the student's mistake.
Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
"""

generation_user_prompt = """
The lesson topic : {topic}

The problem you are student is solving is : {problem}

The student made a following mistake in solving the problem: {student_mistake}

The commonly used strategies and intentions are as follows:
- strategy: {strategy}
- intention: {intention}

Your response should be at most 2 sentences long.
"""

In [29]:
import asyncio
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from tenacity import retry, stop_after_attempt, wait_random_exponential
from langchain_google_genai import ChatGoogleGenerativeAI # https://ai.google.dev/gemini-api/docs/learnlm?hl=ko - LearnLM is integrated into gemini2.5 / https://cloud.google.com/solutions/learnlm/ https://arxiv.org/pdf/2507.06261v1
from aiolimiter import AsyncLimiter


def make_ollama_model(model_tag: str) -> ChatOllama:
    return ChatOllama(
        model=model_tag,
        temperature=0.99,
        num_ctx=4096,  # Explicitly restrict the context size. Increase only if the complete prompt exceeds 4096 tokens.
        keep_alive="10m",  # Keep the currently processed model loaded between requests. It will be explicitly unloaded after all contexts are completed.
    )


def get_model_rate_limiter(model_name):
    limiters ={
        "gemini-3.5-flash-lite": (50, 1), # 50req/s = 3000 rpm // max 4000rpm
        "gemini-3.6-flash": (15, 1), # 15req/s = 900 rpm // max 1000 rpm
        "gpt-5.6-terra": (150, 1), # 150req/s = 9000rpm // max 10000rpm
    }

    if model_name not in limiters:
        return None

    n_request, seconds = limiters[model_name]
    return AsyncLimiter(n_request, seconds)


def get_model_list():
    models = {
        # API model (replace with GPT-5.6 Terra when required)
        "gpt-5.6-terra": ChatOpenAI(model="gpt-5.6-terra", temperature=1.0), # 고성능 상용모델
        # "gemini-3.6-flash": ChatGoogleGenerativeAI(model="gemini-3.6-flash", thinking_level="minimal"), # 저비용 상용 모델 / gemini-2.5-flash 지원 안함, temparature 등 고정값 사용 / https://ai.google.dev/gemini-api/docs/latest-model
        "gemini-3.5-flash-lite": ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", thinking_level="minimal"), # 저비용 상용 모델 / gemini-2.5-flash 지원 안함, temparature 등 고정값 사용 / https://ai.google.dev/gemini-api/docs/latest-model

        # open source models
        # "gemma3:12b": ChatOllama(model="gemma3:12b", temperature=1.0, top_p=0.95, top_k=64, reasoning=False), # 중상급 모델 https://ollama.com/library/gemma4
        "mistral-nemo:12b": ChatOllama(model="mistral-nemo:12b", temperature=1.0, num_ctx=4096), # 중상급 모델 https://ollama.com/library/mistral-nemo
        "llama3.1-8b": ChatOllama(model="llama3.1:8b", temperature=1.0, num_ctx=4096), # 중형모델
        "llama3.2:3b": ChatOllama(model="llama3.2:3b", temperature=1.0, num_ctx=4096), # light weight model
        "llama3.2:1b": ChatOllama(model="llama3.2:1b", temperature=1.0, num_ctx=4096), # super light weight model
    }
    return models


def get_generation_plan():
    """
    The order in this list is the actual execution order.
    Gemma completes all contexts and rounds first.
    Then Llama 8B completes all contexts and rounds.
    Then Llama 3B, Llama 1B, and finally the API model.
    """
    models = get_model_list()
    generation_plan = [
        ("gemini-3.5-flash-lite", models["gemini-3.5-flash-lite"]),

        # ("gemma3:12b", models["gemma3:12b"]),
        ("mistral-nemo:12b", models["mistral-nemo:12b"]),
        ("llama3.1-8b", models["llama3.1-8b"]),
        ("llama3.2:3b", models["llama3.2:3b"]),
        ("llama3.2:1b", models["llama3.2:1b"]),

        ("gpt-5.6-terra", models["gpt-5.6-terra"]),
        # ("gemini-3.6-flash", models["gemini-3.6-flash"]),

    ]
    return generation_plan


def get_generation_prompt():
    prompt = ChatPromptTemplate.from_messages([
        ("system", generation_system_prompt),
        ("human", generation_user_prompt),
    ])
    return prompt


def get_prompt_input(context_data):
    """
    generation_prompt itself does not need to be included in prompt_input.
    This dictionary should contain only variables that actually appear inside generation_system_prompt or generation_user_prompt.
    """
    prompt_input = {
        "topic": context_data["topic"],
        "problem": context_data["problem"],
        # "reference_solution": context_data["reference_solution"],
        "student_mistake": context_data["student_mistake"],
        "error_description": context_data["error_description"],
        "strategy": strategy_dict.get(context_data["strategy"]),
        "intention": intention_dict.get(context_data["intention"]),
    }
    return prompt_input


# 3. Retry and single-response generation
def print_retry(retry_state):
    sleep_time = retry_state.next_action.sleep if retry_state.next_action is not None else 0.0
    error = retry_state.outcome.exception() if retry_state.outcome is not None else "Unknown error"
    print(f"[Retry] attempt={retry_state.attempt_number} | error={error} | sleep={sleep_time:.1f}s", flush=True)


@retry(wait=wait_random_exponential(min=1, max=15), stop=stop_after_attempt(3), reraise=True, before_sleep=print_retry)
async def generate_response(model_name, chain, prompt_input, rate_limiter=None):

    if rate_limiter is not None:
        async with rate_limiter:
            print("---------------rate_limiter applied---------------")
            response = await chain.ainvoke(prompt_input)
    else:
        response = await chain.ainvoke(prompt_input)

    if model_name == "gemini-3.6-flash" or model_name == "gemini-3.5-flash-lite":
        return {
            "model_name": model_name,
            "response": normalize_text(response.text), # gemini returns a list of content blocks. use response.text https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
        }
    else:
        return {
            "model_name": model_name,
            "response": normalize_text(response.content),
        }


async def _run_single_model_job(context_idx, generation_round, data, model_name, chain, semaphore, rate_limiter=None):
    """
    Generates exactly one response: one model x one context x one generation round
    """
    async with semaphore:
        prompt_input = get_prompt_input(data)
        result = await generate_response(model_name=model_name, chain=chain, prompt_input=prompt_input, rate_limiter=rate_limiter)

    return context_idx, generation_round, result


# 4. local Ollama model
async def unload_ollama_model(model):
    """
    Explicitly unloads the current Ollama model before the next local model is processed.
    """
    model_tag = getattr(model, "model", None)
    if not model_tag:
        return
    try:
        process = await asyncio.create_subprocess_exec("ollama", "stop", model_tag, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
        _, stderr = await process.communicate()

        if process.returncode == 0:
            print(f"[Model unload] {model_tag}", flush=True)
        else:
            error_message = stderr.decode("utf-8", errors="replace").strip()
            print(f"[Model unload warning] {model_tag} | {error_message}", flush=True)

    except FileNotFoundError:
        print(f"[Model unload warning] Ollama CLI was not found. Run manually: ollama stop {model_tag}", flush=True)



async def _generate_all_jobs_for_one_model(
    *,
    model_idx,
    model_name,
    model,
    context_data,
    num_generation_response,
    generation_prompt,
    context_round_results,
    max_parallel_requests,
    global_completed_before,
    global_total_jobs,
    pipeline_start_time
    ):
    """
    Model-major execution:
        current model
            -> Context 0, Round 0
            -> Context 0, Round 1
            -> ...
            -> Context 1, Round 0
            -> ...
            -> Last Context, Last Round

    The result is stored in:
        context_round_results
            [context_idx]
            [generation_round]
            [model_idx]
    """
    chain = generation_prompt | model
    semaphore = asyncio.Semaphore(max_parallel_requests)

    rate_limiter = get_model_rate_limiter(model_name)

    jobs = [
        asyncio.create_task(
            _run_single_model_job(
                context_idx=context_idx,
                generation_round=generation_round,
                data=data,
                model_name=model_name,
                chain=chain,
                semaphore=semaphore,
                rate_limiter=rate_limiter
            )
        )
        for context_idx, data in enumerate(context_data)
        for generation_round in range(num_generation_response)
    ]

    model_total_jobs = len(jobs)
    model_completed_jobs = 0

    model_start_time = time.time()

    try:
        for task in asyncio.as_completed(jobs):
            context_idx, generation_round, result = await task
            context_round_results[context_idx][generation_round][model_idx] = result

            model_completed_jobs += 1
            global_completed = global_completed_before + model_completed_jobs

            model_elapsed = int(time.time() - model_start_time)
            model_response_per_second = model_completed_jobs / model_elapsed if model_elapsed > 0 else 0
            model_response_per_minute = (model_completed_jobs * 60) / model_elapsed if model_elapsed > 0 else 0

            total_elapsed = int(time.time() - pipeline_start_time)
            hours, remainder = divmod(total_elapsed, 3600)
            minutes, seconds = divmod(remainder, 60)
            print(
                f"[{model_name}] "
                f"{model_completed_jobs}/{model_total_jobs} "
                f"({100 * model_completed_jobs / model_total_jobs:.1f}%) "
                f"| Overall "
                f"{global_completed}/{global_total_jobs} "
                f"({100 * global_completed / global_total_jobs:.1f}%) "
                f"| Elapsed: {hours}h {minutes}m {seconds}s "
                f"| RPS: {model_response_per_second:.2f} responses/s"
                f"| RPM: {model_response_per_minute:.2f} responses/m",
                flush=True,
            )

    except Exception:
        for task in jobs:
            if not task.done():
                task.cancel()
        await asyncio.gather(*jobs, return_exceptions=True)
        raise

    return model_completed_jobs

In [30]:
import sys
print(sys.executable)

/usr/bin/python3


In [31]:
# results = await a_generate_6_candidates(context_data[0])
# results

## 3. Pedagogy Label Prediction Function ( Probing, Focus, Telling, Generic )
Each category with probabilities

#### Settings

In [32]:
import re
import numpy as np
import torch

from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    RobertaForSequenceClassification,
    DataCollatorWithPadding,
)

MODEL_ID = "dam_tutor_only_best_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.truncation_side = "left"

model = RobertaForSequenceClassification.from_pretrained(MODEL_ID)
model.to(device)
model.eval()


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
    # pad_to_multiple_of=8,  # CUDA Tensor Core 사용 시 선택적으로 고려
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

#### label2id & id2label

In [33]:
# map labels
def label2id_id2label(label_names: list, show_stats: bool = True) -> dict:
    """
        map labels to integers.
        converted labels are stored in a new column called 'labels'.
    """
    # label_names = sorted(df["label"].unique())

    label2id = {label: i for i, label in enumerate(label_names)}
    id2label = {i: label for label, i in label2id.items()}

    if show_stats:
        print("[label_names]")
        print(label_names)

        print("\n[label2id]")
        print(label2id)

        print("\n[id2label]")
        print(id2label)

    return label2id, id2label


In [34]:
label_names = ['focus', 'generic', 'probing', 'telling']
num_labels = len(label_names)
label2id, id2label = label2id_id2label(label_names)

[label_names]
['focus', 'generic', 'probing', 'telling']

[label2id]
{'focus': 0, 'generic': 1, 'probing': 2, 'telling': 3}

[id2label]
{0: 'focus', 1: 'generic', 2: 'probing', 3: 'telling'}


#### Prediction Function
코드 설명 : https://chatgpt.com/s/t_6a7a866db5dc8191b4ca5c3c1dd5705f

In [35]:
def batch_predict_pedagogy( batch_texts: list[str], top_k: int = 4, batch_size: int = 6):

    model.eval()  # Dropout은 model.eval()을 하지 않으면 계속 활성화될 수 있습니다. PyTorch도 inference 시 model.eval()과 torch.no_grad()를 별도로 사용하도록 설명합니다. https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html?utm_source=chatgpt.com
    tokenized_dataset = [ tokenizer(normalize_text(text), truncation=True) for text in batch_texts ]

    dataloader = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collator)

    all_probs = []

    with torch.no_grad(): # inference mode
        for batch in dataloader: # [B, max_length in batch]
            batch = { key: value.to(device) for key, value in batch.items() }
            logits = model(**batch).logits        # [B, num_labels]
            probs = torch.softmax(logits, dim=-1) # apply sofmax to logits in the last dimention [B, num labels] -> softmax on num labels

            all_probs.append(probs.cpu()) # [ [B_1, num_labels], [B_2, num_labels], ... ] - num_labels 개의 확률 값

    # (N, num_labels)
    all_probs = torch.cat(all_probs, dim=0) # concatenate the tensors in the first dimention -> [ B_1 + B_2 + ... + B_N, num_labels ]

    # 각 text마다 확률이 높은 top_k개
    top_probs, top_ids = torch.topk(all_probs, k=top_k, dim=1) # [ B_1 + B_2 + ... + B_N, top_k = 4 ] -> num_label 방향으로 높은 확률로 순서대로 가져옴 / top_probs = 각 4개 확률, top_ids = 각 4개 레이블 인덱스 / 행수는 B_1 + B_2 + ... + B_N

    pred_results = []

    for label_ids, probs in zip(top_ids, top_probs):
        predictions = []

        for label_id, prob in zip(label_ids, probs):
            predictions.append({
                "label": id2label[label_id.item()],
                "probability": round(prob.item(), 6)
            })

        pred_results.append({
            "final_label": id2label[label_ids[0].item()],           # not necessary, but for the sake of clarity
            "final_prob": round(probs[0].item(), 6),                # not necessary, but for the sake of clarity
            "prediction": predictions
        })

    return pred_results # list of dict. len(pred_results) = len(batch_texts)

## 4. Generate tutor responses & predict the labels of responese per batch

( 6 responses per batch, generated by different models )

In [36]:
n_models = 4

In [37]:
from pprint import pprint
import asyncio
import time

async def generate_tutor_responses_and_predict_labels_per_batch(
    context_data,
    num_generation_response=6,
    local_max_parallel_requests=1,
    api_max_parallel_requests=3,
    prediction_chunk_size=32,
    prediction_batch_size=32,
    unload_local_after_each_model=True,
):
    start_time = time.time()

    num_contexts = len(context_data)
    generation_prompt = get_generation_prompt()
    generation_plan = get_generation_plan()
    n_models = len(generation_plan)
    total_generation_jobs = num_contexts * num_generation_response * n_models

    print("Starting model-major generation & prediction pipeline")
    print(
        f"Number of Contexts             : {num_contexts}\n"
        f"Number of Batches per context  : {num_generation_response}\n"
        f"Number of Models               : {n_models}\n"
        f"Total Number of Responses      : {total_generation_jobs}\n"
        f"Local parallel requests/model  : {local_max_parallel_requests}\n"
        f"API parallel requests/model    : {api_max_parallel_requests}\n"
        f"Prediction Chunk Size          : {prediction_chunk_size}\n"
        f"Prediction Batch Size          : {prediction_batch_size}\n"
    )

    ####################################
    # Step 1. Prompt preparation
    ####################################
    formatted_prompts = []

    for data in context_data:
        prompt_input = get_prompt_input(data)
        formatted_prompt = generation_prompt.format(**prompt_input)
        formatted_prompts.append(formatted_prompt)

    print(f"[Step 1/5] Prompt preparation completed ({num_contexts} prompts)\n")

    ####################################
    # Step 2. Result storage, Shape: [context][round][model]
    ####################################
    context_round_results = [[[None] * n_models for _ in range(num_generation_response)] for _ in range(num_contexts)]

    ####################################
    # Step 2. Model-major generation
    ####################################
    print("[Step 2/5] Starting model-major response generation")

    completed_generation_jobs = 0

    for model_idx, (model_name, model) in enumerate(generation_plan):
        is_local_model = isinstance(model, ChatOllama)

        if is_local_model:
            max_parallel_requests = local_max_parallel_requests
            model_kind = "local Ollama"
        else:
            max_parallel_requests = api_max_parallel_requests
            model_kind = "API"

        print(f"\n[Model {model_idx + 1}/{n_models}] {model_name} | type={model_kind} | parallel_requests={max_parallel_requests}", flush=True,)

        try:
            completed_for_model = await _generate_all_jobs_for_one_model(
                model_idx=model_idx,
                model_name=model_name,
                model=model,
                context_data=context_data,
                num_generation_response=num_generation_response,
                generation_prompt=generation_prompt,
                context_round_results=context_round_results,
                max_parallel_requests=max_parallel_requests,
                global_completed_before=completed_generation_jobs,
                global_total_jobs=total_generation_jobs,
                pipeline_start_time=start_time
            )
            completed_generation_jobs += completed_for_model
        finally:
            # This also executes when generation raises an error.
            if is_local_model and unload_local_after_each_model:
                await unload_ollama_model(model)

    print(f"\n[Step 2/5] Generation completed ({completed_generation_jobs} responses)\n")

    # Check for missing results
    missing_slots = [
        (context_idx, generation_round, model_idx)
        for context_idx, round_results in enumerate(context_round_results)
        for generation_round, model_results in enumerate(round_results)
        for model_idx, result in enumerate(model_results)
        if result is None
    ]
    if missing_slots:
        raise RuntimeError(
            f"{len(missing_slots)} generation slots are missing. "
            f"First missing slots: {missing_slots[:10]}"
        )

    ####################################
    # Step 3. Flatten generation results
    ####################################
    print("[Step 3/5] Flattening generation results...")
    flat_results_per_context = []
    all_flat_generation_results = []
    for round_results in context_round_results:
        flat_results = [result for model_results in round_results for result in model_results]
        flat_results_per_context.append(flat_results)
        all_flat_generation_results.extend(flat_results)
    all_llm_responses = [result["response"] for result in all_flat_generation_results]
    print(
        f"[Step 3/5] Flatten completed\n"
        f"- total contexts: {num_contexts}\n"
        f"- total responses: {len(all_llm_responses)}\n"
    )

    ####################################
    # Step 4. Pedagogy prediction
    ####################################
    print("[Step 4/5] Starting pedagogy prediction...")
    all_pred_results = []
    total_responses = len(all_llm_responses)
    total_prediction_chunks = (total_responses + prediction_chunk_size - 1) // prediction_chunk_size
    for chunk_idx, start_idx in enumerate(
        range(0, total_responses, prediction_chunk_size),
        start=1,
    ):
        end_idx = min(start_idx + prediction_chunk_size, total_responses)
        response_chunk = all_llm_responses[start_idx:end_idx]
        pred_results = batch_predict_pedagogy(
            response_chunk,
            top_k=4,
            batch_size=prediction_batch_size,
        )
        all_pred_results.extend(pred_results)
        print(
            f"[Prediction Chunk {chunk_idx}/{total_prediction_chunks}] Completed | "
            f"{end_idx}/{total_responses} responses processed",
            flush=True,
        )
    if len(all_pred_results) != len(all_flat_generation_results):
        raise RuntimeError(
            "Prediction count does not match generation count: "
            f"{len(all_pred_results)} predictions vs {len(all_flat_generation_results)} generations"
        )
    print()

    ####################################
    # Step 5. Add prediction results
    ####################################
    print("[Step 5/5] Building final results...")
    for generation_result, pred_result in zip(all_flat_generation_results, all_pred_results):
        generation_result["pred_result"] = pred_result


    ####################################
    # Final result structure
    ####################################
    all_results = []
    for context_idx, data in enumerate(context_data):
        round_results = context_round_results[context_idx]
        generations = [{"generation_round": generation_round, "round_generation_results": model_results}
                      for generation_round, model_results in enumerate(round_results)]
        strategy = strategy_dict.get(data["strategy"])
        intention = intention_dict.get(data["intention"])
        all_results.append({
            "topic": data["topic"],
            "problem": data["problem"],
            "error_category": data["error_category"],
            "student_mistake": data["student_mistake"],
            "context_data_idx": context_idx,
            "prompt": formatted_prompts[context_idx],
            "strategy": strategy,
            "intention": intention,
            "batch_generation_results": generations,
        })
        if (context_idx + 1) % 100 == 0 or context_idx + 1 == num_contexts:
            print(
                f"[Finalizing] {context_idx + 1}/{num_contexts} contexts "
                f"({100 * (context_idx + 1) / num_contexts:.1f}%)",
                flush=True,
            )


    ####################################
    # Final summary
    ####################################
    total_elapsed = int(time.time() - start_time)
    hours, remainder = divmod(total_elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\nPipeline completed")
    print("-" * 90)
    print(
        f"Number of Contexts             : {num_contexts}\n"
        f"Number of Models               : {n_models}\n"
        f"Total Generation Jobs          : {total_generation_jobs}\n"
        f"Total Generated Responses      : {len(all_llm_responses)}\n"
        f"Total Predictions              : {len(all_pred_results)}\n"
        f"Total Elapsed Time             : {hours}h {minutes}m {seconds}s"
    )
    print("=" * 90)
    return all_results

In [38]:
# !nvidia-smi
# !tail -n 100 /content/ollama_serve.log
# !dmesg -T | tail -n 50

In [39]:
all_results = None

if os.path.exists(ALL_RESULTS_FILE_PATH):
    print(f"Loading {ALL_RESULTS_FILE_PATH}...")
    with open(ALL_RESULTS_FILE_PATH, 'r', encoding='utf-8') as f:
        all_results = json.load(f)
else:
    print(f'adding strategy and intention to context data...')
    all_results = await generate_tutor_responses_and_predict_labels_per_batch(
        # shuffled_context_data,
        context_data=context_data_with_strategy_intention,
        num_generation_response=12,

        # Only affects local models:
        local_max_parallel_requests=100, # OLLAMA_NUM_PARALLEL=100

        # Only affects API models:
        api_max_parallel_requests=200,

        prediction_chunk_size=32,
        prediction_batch_size=32,

        unload_local_after_each_model=True,
    )

    os.makedirs(os.path.dirname(ALL_RESULTS_FILE_PATH), exist_ok=True)

    with open(ALL_RESULTS_FILE_PATH, 'w', encoding='utf-8') as f:
        print(f"Saving {ALL_RESULTS_FILE_PATH}...")
        json.dump(all_results, f, ensure_ascii=False, indent=2)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
[gpt-5.6-terra] 6511/8316 (78.3%) | Overall 48091/49896 (96.4%) | Elapsed: 0h 42m 35s | RPS: 86.81 responses/s| RPM: 5208.80 responses/m
---------------rate_limiter applied---------------
[gpt-5.6-terra] 6512/8316 (78.3%) | Overall 48092/49896 (96.4%) | Elapsed: 0h 42m 35s | RPS: 86.83 responses/s| RPM: 5209.60 responses/m
---------------rate_limiter applied---------------
[gpt-5.6-terra] 6513/8316 (78.3%) | Overall 48093/49896 (96.4%) | Elapsed: 0h 42m 35s | RPS: 86.84 responses/s| RPM: 5210.40 responses/m
---------------rate_limiter applied---------------
[gpt-5.6-terra] 6514/8316 (78.3%) | Overall 48094/49896 (96.4%) | Elapsed: 0h 42m 35s | RPS: 86.85 responses/s| RPM: 5211.20 responses/m
---------------rate_limiter applied---------------
[gpt-5.6-terra] 6515/8316 (78.3%) | Overall 48095/49896 (96.4%) | Elapsed: 0h 42m 35s | RPS: 86.87 responses/s| RPM: 5212.00 responses/m
---------------rate_limiter applied---------------
[gpt-5.6-terra] 6516/831

In [11]:
if os.path.exists(ALL_RESULTS_FILE_PATH):
    print(f"Loading {ALL_RESULTS_FILE_PATH}...")
    with open(ALL_RESULTS_FILE_PATH, 'r', encoding='utf-8') as f:
        all_results = json.load(f)

Loading data_preference_train_only/bridge_2tage__with_reference_solution_more_models/preference_score__all_results.json...


## Preference Dataset Construction
- PMI itself can lead to a zero-devision error thus here smoothing factor of 10^-12 is added to the denominator
- Postive PMI is a good alternative. However it only considers the postivie correlation between the error category and pedagogy labels, and does not give penaly the negative correlation(rare cases, irrelevance) between them.
- NPMI : PMI itself gives a huge weights to rare occurances thus normalizing the scores into [0,1] range.

#### Get point wise mutual information table ( Normalized Pointwise Mutual Information )

https://web.stanford.edu/~jurafsky/slp3/J.

https://homepages.inf.ed.ac.uk/sgwater/teaching/lsa2015/labs/lab4-sol.html


In [12]:
ppmi_table = pd.read_csv("data_pmi_table/ppmi_table.csv", index_col=0)
ppmi_table

,calculation_error_easily_solved_by_a_calculator,extra_quantity_or_missing_quantity,missing_wrong_factual_knowledge,misunderstanding_of_a_question,none_of_the_above,reached_correct_solution_but_proceeded_further,unit_conversion_error
pedagogy,,,,,,,
focus,0.898611,0.000000,0.282210,0.000000,0.297160,0.267413,0.000000
generic,0.000000,0.050277,0.000000,0.074891,0.037286,0.000000,0.057012
probing,0.000000,0.000000,0.474229,0.000000,0.000000,0.237039,0.179324
telling,0.000000,0.468045,1.666874,0.000000,0.000000,0.000000,1.179324


In [13]:
# pmi_table = pd.read_csv("data_pmi_table/npmi_table.csv", index_col=0)
# pmi_table

In [14]:
# npmi_table = pd.read_csv("data_pmi_table/npmi_table.csv", index_col=0)
# # npmi_table

In [15]:
all_categories = [data['error_category'] for data in data]
unique_error_categories = list(set(all_categories))

# Check if all unique_error_categories are in npmi_table.columns
all_in_columns = all([cat in ppmi_table.columns for cat in unique_error_categories])
print("All unique_error_categories are in npmi_table.columns:", all_in_columns)

# Check if all npmi_table.columns are in unique_error_categories
all_columns_in_categories = all([col in unique_error_categories for col in ppmi_table.columns])
print("All npmi_table.columns are in unique_error_categories:", all_columns_in_categories)

All unique_error_categories are in npmi_table.columns: True
All npmi_table.columns are in unique_error_categories: True


## Compute response scores

In [16]:
def compute_scores(data=all_results):
    for per_context_result in all_results:
        error_category = per_context_result['error_category']


        per_batch_round_preference_score_results = []
        for generation_results_per_batch in per_context_result['batch_generation_results']:
            generation_round = generation_results_per_batch['generation_round'] # index of current batch

            scores = []
            for round_result in generation_results_per_batch['round_generation_results']:
                pedagogy_label_and_probs = round_result['pred_result']['prediction']
                score = 0.0

                for label_prob in pedagogy_label_and_probs:
                    label = label_prob['label']
                    prob = label_prob['probability']

                    # pmi_score = npmi_table.loc[label, error_category] # npmi
                    pmi_score = ppmi_table.loc[label, error_category] # ppmi
                    score += prob * pmi_score
                    # pmi_scores.append(pmi_score)

                round_result['score'] = score

    return all_results


all_results = compute_scores(all_results)

In [17]:
with open(ALL_RESULTS_FILE_PATH, 'w', encoding='utf-8') as f:
    print(f"Saving {ALL_RESULTS_FILE_PATH}...")
    json.dump(all_results, f, ensure_ascii=False, indent=2)

Saving data_preference_train_only/bridge_2tage__with_reference_solution_more_models/preference_score__all_results.json...


## Convert Results into a Dataframe 

In [18]:
all_results = json.load(open(ALL_RESULTS_FILE_PATH, 'r', encoding='utf-8'))

In [19]:
def get_result_dataframe(data=all_results):
    rows = []

    for context in all_results:

        context_idx = context["context_data_idx"]
        error_category = context["error_category"]
        topic = context["topic"]
        problem = context["problem"]
        generation_prompt = context["prompt"]
        student_mistake = context["student_mistake"]

        if "strategy" in context and "intention" in context:
            strategy = context["strategy"]
            intention = context["intention"]
        else:
            strategy = None
            intention = None

        for batch_result in context["batch_generation_results"]:
            for round_result in batch_result["round_generation_results"]:

                row = {
                    "context_data_idx": context_idx,
                    "generation_round": batch_result["generation_round"],
                    "problem": problem,
                    "topic": topic,
                    "student_mistake": student_mistake,
                    "error_category": error_category,

                    "predicted_pedagogy": round_result["pred_result"]["final_label"],
                    "predicted_category_prob": round_result["pred_result"]["final_prob"],
                    "predicted_category_prob_list": round_result["pred_result"]["prediction"],

                    "prompt": generation_prompt,
                    "model": round_result["model_name"],
                
                    "response": round_result["response"],
                    "score": round_result["score"],
                    "response_word_count": len(round_result["response"].split()),
                }

                # strategy/intention이 존재하는 데이터인 경우에만 추가
                if strategy is not None and intention is not None:
                    row["strategy"] = strategy
                    row["intention"] = intention

                rows.append(row.copy())



    result_dataframe = pd.DataFrame(rows)
    before_length = len(result_dataframe)
    result_dataframe = result_dataframe.drop_duplicates(subset=["response"])
    after_length = len(result_dataframe)
    print('before drop_duplicates', before_length)
    print('after drop_duplicates', after_length)
    print('drop_duplicates', before_length - after_length)

    return result_dataframe

In [20]:
all_results_df = get_result_dataframe(all_results)
all_results_df.head(1)

before drop_duplicates 49896
after drop_duplicates 48839
drop_duplicates 1057


,context_data_idx,generation_round,problem,topic,student_mistake,error_category,predicted_pedagogy,predicted_category_prob,predicted_category_prob_list,prompt,model,response,score,response_word_count,strategy,intention
0,0,0,milly is making feather boas for her dance tea...,Math Word Problem,"'milly can only take 25 of 20 tail feathers, w...",reached_correct_solution_but_proceeded_further,focus,0.62298,"[{'label': 'focus', 'probability': 0.62298}, {...",System: \nYou are an experienced elementary ma...,gemini-3.5-flash-lite,you figured out the total number of feathers m...,0.253611,59,provide a minor correction,"explain the student's mistake (e.g., what is w..."


In [21]:
all_results_df.to_csv(ALL_RESULTS_DF_FILE_PATH, index=False)

## Construct Preference Pairs ( with all metadata )

The first n_pair rows are paired with the last n_pair rows within the same context_data_idx ( same prompt )

In [22]:
for context_data_idx, context_df in all_results_df.groupby("context_data_idx", sort=False):
    print(context_df)
    break

    context_data_idx  generation_round  \
0                  0                 0   
1                  0                 0   
2                  0                 0   
3                  0                 0   
4                  0                 0   
..               ...               ...   
67                 0                11   
68                 0                11   
69                 0                11   
70                 0                11   
71                 0                11   

                                              problem              topic  \
0   milly is making feather boas for her dance tea...  Math Word Problem   
1   milly is making feather boas for her dance tea...  Math Word Problem   
2   milly is making feather boas for her dance tea...  Math Word Problem   
3   milly is making feather boas for her dance tea...  Math Word Problem   
4   milly is making feather boas for her dance tea...  Math Word Problem   
..                                     

In [23]:
def make_response_pairs(all_results_df):
    all_response_pairs = []
    for context_data_idx, context_df in all_results_df.groupby("context_data_idx", sort=False):

        context_df = context_df.sort_values("score", ascending=False).reset_index(drop=True)
        n_pairs = len(context_df) // 2

        if n_pairs == 0:
            continue

        chosen_df = context_df.iloc[:n_pairs].reset_index(drop=True) # first n_pair rows with the same context_data_idx
        rejected_df = context_df.iloc[-n_pairs:].iloc[::-1].reset_index(drop=True) # last n_pair rows with the same context_data_idx

        # these data are identical within the same context_data_idx 
        context_data_idx = context_df["context_data_idx"].iloc[0]
        prompt = context_df["prompt"].iloc[0]
        error_category = context_df["error_category"].iloc[0]
        problem = context_df["problem"].iloc[0]
        topic = context_df["topic"].iloc[0]
        student_mistake = context_df["student_mistake"].iloc[0]

        context_pairs = pd.DataFrame({
            "context_data_idx": [context_data_idx] * n_pairs,
            "error_category": [error_category] * n_pairs,
            "problem": [problem] * n_pairs,
            "topic": [topic] * n_pairs,
            "student_mistake": [student_mistake] * n_pairs,
            "prompt": [prompt] * n_pairs,

            "chosen_model": chosen_df["model"],
            "chosen_response": chosen_df["response"].to_numpy(),
            "chosen_response_word_count": chosen_df["response_word_count"].to_numpy(),
            "chosen_intention": chosen_df["intention"],
            "chosen_strategy": chosen_df["strategy"],
            "chosen_score": chosen_df["score"].to_numpy(),
            "chosen_label": chosen_df["predicted_pedagogy"],
            "chosen_label_prob": chosen_df["predicted_category_prob"],
            "chosen_label_prob_list": chosen_df["predicted_category_prob_list"],

            "rejected_model": rejected_df["model"],
            "rejected_response": rejected_df["response"].to_numpy(),
            "rejected_response_word_count": rejected_df["response_word_count"].to_numpy(),
            "rejected_intention": rejected_df["intention"],
            "rejected_strategy": rejected_df["strategy"],
            "rejected_score": rejected_df["score"].to_numpy(),
            "rejected_label": rejected_df["predicted_pedagogy"],
            "rejected_label_prob": rejected_df["predicted_category_prob"],
            "rejected_label_prob_list": rejected_df["predicted_category_prob_list"],
        })

        all_response_pairs.append(context_pairs)

    if not all_response_pairs:
        return pd.DataFrame(columns=[
            "context_data_idx",
            "prompt",
            "chosen_response",
            "chosen_score",
            "rejected_response",
            "rejected_score",
        ])

    response_pairs = pd.concat(all_response_pairs, ignore_index=True)
    response_pairs["score_difference"] = response_pairs["chosen_score"] - response_pairs["rejected_score"]
    response_pairs = response_pairs[response_pairs["chosen_response"] != response_pairs["rejected_response"]]

    # merge strategy and intention into a single column ( same strategy & intention )
    response_pairs['strategy'] = response_pairs.apply(lambda row: row['chosen_strategy'] if row['chosen_strategy'] == row['rejected_strategy'] else None, axis=1)
    response_pairs['intention'] = response_pairs.apply(lambda row: row['chosen_intention'] if row['chosen_intention'] == row['rejected_intention'] else None, axis=1)

    # Normalize the score difference by the maximum PPMI for each error category
    max_ppmi_per_error = ppmi_table.max(axis=0)  # 1. Get the maximum PPMI for each error category
    response_pairs["score_difference"]  # 2. Each preference pair's raw margin
    response_pairs["max_ppmi"] = (response_pairs["error_category"].map(max_ppmi_per_error))  # 3. Map each pair's error category to its max PPMI
    response_pairs["normalized_score_difference"] = (response_pairs["score_difference"] / response_pairs["max_ppmi"])  # 4. Calculate the normalized margin

    # Drop chosen_strategy, chosen_intention, rejected_strategy, rejected_intention columns
    columns_to_drop = ['chosen_strategy', 'chosen_intention', 'rejected_strategy', 'rejected_intention']
    response_pairs = response_pairs.drop(columns=columns_to_drop, errors='ignore')

    return response_pairs

In [24]:
response_pairs_df = make_response_pairs(all_results_df)

In [25]:
response_pairs_df.to_csv(RESPONSE_PAIRS_DF_FILE_PATH, index=False)

In [26]:
import pandas as pd
from pprint import pprint
response_pairs_df=pd.read_csv(RESPONSE_PAIRS_DF_FILE_PATH).to_dict(orient="records")

pprint(response_pairs_df[0])

{'chosen_label': 'focus',
 'chosen_label_prob': 0.922704,
 'chosen_label_prob_list': "[{'label': 'focus', 'probability': 0.922704}, "
                           "{'label': 'probing', 'probability': 0.060051}, "
                           "{'label': 'telling', 'probability': 0.015874}, "
                           "{'label': 'generic', 'probability': 0.001371}]",
 'chosen_model': 'llama3.2:3b',
 'chosen_response': 'the student is correct in calculating the total number of '
                    'feathers needed, but made a mistake in converting the '
                    'total feathers to the number of feathers per flamingo by '
                    "using the decimal equivalent of 0.25 as 5, which doesn't "
                    'accurately represent the original instruction to pluck 25 '
                    'feathers at a time from each flamingo. let me try a '
                    'different approach. to get to the answer, you need to '
                    'find out how many feathers mill

----